# Ensemble Stacking for Car Damage Severity Classification
# ===========================================
# 
# This notebook implements ensemble stacking using multiple pre-trained models
# to improve classification performance on car damage severity detection.
#
# **Stacking Strategy:**
# - Base Models: VGG16, VGG19, ResNet50, DenseNet121, MobileNetV2
# - Meta-Learner: Logistic Regression / XGBoost / Neural Network
# - Cross-Validation: Stratified K-Fold to prevent overfitting


# 1. Imports and Setup


In [ ]:
import os
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import (
    VGG16, VGG19, ResNet50, DenseNet121, MobileNetV2
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Scikit-learn
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, f1_score, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder

# XGBoost (if available)
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available. Using Logistic Regression as meta-learner.")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


# 2. Configuration and Paths


In [ ]:
# Dataset paths (Kaggle format)
TRAIN_DIR = '/kaggle/input/car-damage-detection/Car_Demage_Detection_Datasets/Car_Demage_Severity/training'
VAL_DIR = '/kaggle/input/car-damage-detection/Car_Demage_Detection_Datasets/Car_Demage_Severity/validation'

# Model weights directory (if using pre-trained models)
MODELS_DIR = '/kaggle/input/car-damage-models'  # Adjust based on your dataset structure
# Alternative: Use local models if available
LOCAL_MODELS_DIR = '../Models_Weight_files'

# Parameters
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
N_SPLITS = 5  # Number of folds for cross-validation
EPOCHS = 50  # Epochs for base models (if training from scratch)

# Meta-learner options
USE_XGBOOST = XGBOOST_AVAILABLE
USE_NEURAL_NET = True  # Use neural network as meta-learner


# 3. Helper Functions


In [ ]:
def create_base_model(model_name, num_classes, weights='imagenet'):
    """
    Create a base model with transfer learning
    """
    base_models = {
        'vgg16': VGG16,
        'vgg19': VGG19,
        'resnet50': ResNet50,
        'densenet121': DenseNet121,
        'mobilenetv2': MobileNetV2
    }
    
    if model_name.lower() not in base_models:
        raise ValueError(f"Unknown model: {model_name}")
    
    # Load base model
    base_model_class = base_models[model_name.lower()]
    base_model = base_model_class(
        weights=weights,
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )
    
    # Freeze base layers
    base_model.trainable = False
    
    # Add custom head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = Dense(256, activation='relu')(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    return model

def load_or_create_model(model_name, num_classes, model_path=None):
    """
    Load pre-trained model or create new one
    """
    if model_path and os.path.exists(model_path):
        print(f"Loading {model_name} from {model_path}")
        return load_model(model_path)
    else:
        print(f"Creating new {model_name} model")
        return create_base_model(model_name, num_classes)

def get_data_generators(train_dir, val_dir, batch_size=32, augment=True):
    """
    Create data generators for training and validation
    """
    if augment:
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            horizontal_flip=True,
            zoom_range=0.2,
            fill_mode='nearest'
        )
    else:
        train_datagen = ImageDataGenerator(rescale=1./255)
    
    val_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=IMG_SIZE,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=True
    )
    
    val_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=IMG_SIZE,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )
    
    return train_generator, val_generator

def get_all_images_and_labels(data_dir):
    """
    Get all image paths and labels from directory
    """
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(data_dir))
    
    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        if os.path.isdir(class_dir):
            for img_file in os.listdir(class_dir):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    image_paths.append(os.path.join(class_dir, img_file))
                    labels.append(class_name)
    
    return image_paths, labels, class_names


# 4. Prepare Data for Stacking


In [ ]:
# Get all images and labels
train_paths, train_labels, class_names = get_all_images_and_labels(TRAIN_DIR)
val_paths, val_labels, _ = get_all_images_and_labels(VAL_DIR)

print(f"Training samples: {len(train_paths)}")
print(f"Validation samples: {len(val_paths)}")
print(f"Classes: {class_names}")
print(f"Number of classes: {len(class_names)}")

# Encode labels
label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)
val_labels_encoded = label_encoder.transform(val_labels)

num_classes = len(class_names)


# 5. Generate Base Model Predictions with Cross-Validation


In [ ]:
# Define base models to use
BASE_MODELS = ['vgg16', 'vgg19', 'resnet50', 'densenet121', 'mobilenetv2']

# Model paths mapping (adjust based on your structure)
MODEL_PATHS = {
    'vgg16': os.path.join(LOCAL_MODELS_DIR, 'vgg16_best.h5') if os.path.exists(LOCAL_MODELS_DIR) else None,
    'vgg19': os.path.join(LOCAL_MODELS_DIR, 'vgg19_best.h5') if os.path.exists(LOCAL_MODELS_DIR) else None,
    'resnet50': os.path.join(LOCAL_MODELS_DIR, 'resnet50_best.h5') if os.path.exists(LOCAL_MODELS_DIR) else None,
    'densenet121': os.path.join(LOCAL_MODELS_DIR, 'densenet121_best.h5') if os.path.exists(LOCAL_MODELS_DIR) else None,
    'mobilenetv2': os.path.join(LOCAL_MODELS_DIR, 'mobilenetv2_best.h5') if os.path.exists(LOCAL_MODELS_DIR) else None,
}

def preprocess_image(img_path):
    """Load and preprocess a single image"""
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    return img_array

def get_predictions_from_model(model, image_paths, batch_size=32):
    """Get predictions from a model for all images"""
    predictions = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_images = np.vstack([preprocess_image(path) for path in batch_paths])
        batch_preds = model.predict(batch_images, verbose=0)
        predictions.append(batch_preds)
    return np.vstack(predictions)


In [ ]:
# Initialize storage for out-of-fold predictions
train_oof_predictions = {}
val_predictions = {}

# Stratified K-Fold for cross-validation
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

print("Generating base model predictions with cross-validation...")
print("=" * 60)

for model_name in BASE_MODELS:
    print(f"\nProcessing {model_name.upper()}...")
    
    # Initialize arrays for OOF predictions
    train_oof_preds = np.zeros((len(train_paths), num_classes))
    val_preds = None
    
    # Load or create model
    model_path = MODEL_PATHS.get(model_name)
    model = load_or_create_model(model_name, num_classes, model_path)
    
    # Cross-validation loop
    for fold, (train_idx, val_idx) in enumerate(skf.split(train_paths, train_labels_encoded)):
        print(f"  Fold {fold + 1}/{N_SPLITS}...", end=' ')
        
        # Get validation predictions for this fold
        val_fold_paths = [train_paths[i] for i in val_idx]
        val_fold_preds = get_predictions_from_model(model, val_fold_paths)
        train_oof_preds[val_idx] = val_fold_preds
        
        print("Done")
    
    # Get predictions on full validation set
    print("  Generating validation predictions...", end=' ')
    val_preds = get_predictions_from_model(model, val_paths)
    print("Done")
    
    # Store predictions
    train_oof_predictions[model_name] = train_oof_preds
    val_predictions[model_name] = val_preds
    
    print(f"  {model_name.upper()} completed!")

print("\n" + "=" * 60)
print("All base model predictions generated!")


# 6. Prepare Stacking Features


In [ ]:
# Combine predictions from all base models
def combine_predictions(predictions_dict):
    """Combine predictions from multiple models into a feature matrix"""
    features = []
    for model_name in BASE_MODELS:
        preds = predictions_dict[model_name]
        features.append(preds)
    return np.hstack(features)

# Create feature matrices
X_train_stack = combine_predictions(train_oof_predictions)
X_val_stack = combine_predictions(val_predictions)
y_train_stack = train_labels_encoded
y_val_stack = val_labels_encoded

print(f"Training stacking features shape: {X_train_stack.shape}")
print(f"Validation stacking features shape: {X_val_stack.shape}")
print(f"Number of features per sample: {X_train_stack.shape[1]} ({len(BASE_MODELS)} models × {num_classes} classes)")


# 7. Train Meta-Learner


In [ ]:
# Option 1: Logistic Regression Meta-Learner
print("Training Logistic Regression Meta-Learner...")
meta_lr = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    random_state=42,
    C=1.0
)
meta_lr.fit(X_train_stack, y_train_stack)

# Predictions
train_pred_lr = meta_lr.predict(X_train_stack)
val_pred_lr = meta_lr.predict(X_val_stack)

train_acc_lr = accuracy_score(y_train_stack, train_pred_lr)
val_acc_lr = accuracy_score(y_val_stack, val_pred_lr)

print(f"Logistic Regression - Train Accuracy: {train_acc_lr:.4f}, Val Accuracy: {val_acc_lr:.4f}")


In [ ]:
# Option 2: XGBoost Meta-Learner (if available)
if USE_XGBOOST and XGBOOST_AVAILABLE:
    print("\nTraining XGBoost Meta-Learner...")
    meta_xgb = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=num_classes,
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric='mlogloss'
    )
    meta_xgb.fit(
        X_train_stack, y_train_stack,
        eval_set=[(X_val_stack, y_val_stack)],
        verbose=False
    )
    
    train_pred_xgb = meta_xgb.predict(X_train_stack)
    val_pred_xgb = meta_xgb.predict(X_val_stack)
    
    train_acc_xgb = accuracy_score(y_train_stack, train_pred_xgb)
    val_acc_xgb = accuracy_score(y_val_stack, val_pred_xgb)
    
    print(f"XGBoost - Train Accuracy: {train_acc_xgb:.4f}, Val Accuracy: {val_acc_xgb:.4f}")
else:
    meta_xgb = None
    print("\nXGBoost not available or disabled.")


In [ ]:
# Option 3: Neural Network Meta-Learner
if USE_NEURAL_NET:
    print("\nTraining Neural Network Meta-Learner...")
    
    # Create simple neural network
    meta_nn = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train_stack.shape[1],)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    meta_nn.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Convert labels to categorical for neural network
    history = meta_nn.fit(
        X_train_stack, y_train_stack,
        validation_data=(X_val_stack, y_val_stack),
        epochs=50,
        batch_size=32,
        verbose=1,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
        ]
    )
    
    train_pred_nn = np.argmax(meta_nn.predict(X_train_stack, verbose=0), axis=1)
    val_pred_nn = np.argmax(meta_nn.predict(X_val_stack, verbose=0), axis=1)
    
    train_acc_nn = accuracy_score(y_train_stack, train_pred_nn)
    val_acc_nn = accuracy_score(y_val_stack, val_pred_nn)
    
    print(f"Neural Network - Train Accuracy: {train_acc_nn:.4f}, Val Accuracy: {val_acc_nn:.4f}")
else:
    meta_nn = None
    print("\nNeural Network meta-learner disabled.")


# 8. Select Best Meta-Learner and Evaluate


In [ ]:
# Compare all meta-learners and select the best one
meta_learners = {
    'Logistic Regression': (meta_lr, val_acc_lr, val_pred_lr),
}

if meta_xgb is not None:
    meta_learners['XGBoost'] = (meta_xgb, val_acc_xgb, val_pred_xgb)

if meta_nn is not None:
    meta_learners['Neural Network'] = (meta_nn, val_acc_nn, val_pred_nn)

# Find best meta-learner
best_meta_name = max(meta_learners.keys(), key=lambda x: meta_learners[x][1])
best_meta_model, best_meta_acc, best_meta_pred = meta_learners[best_meta_name]

print(f"\n{'='*60}")
print(f"Best Meta-Learner: {best_meta_name}")
print(f"Validation Accuracy: {best_meta_acc:.4f}")
print(f"{'='*60}\n")

# Detailed evaluation
print("Classification Report:")
print(classification_report(y_val_stack, best_meta_pred, target_names=class_names))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val_stack, best_meta_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix - {best_meta_name} Stacking')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Compare individual models vs ensemble
print("\n" + "="*60)
print("Model Comparison:")
print("="*60)

# Calculate individual model accuracies
for model_name in BASE_MODELS:
    individual_preds = np.argmax(val_predictions[model_name], axis=1)
    individual_acc = accuracy_score(y_val_stack, individual_preds)
    print(f"{model_name.upper():20s}: {individual_acc:.4f}")

print(f"{'STACKING (' + best_meta_name + ')':20s}: {best_meta_acc:.4f}")
print("="*60)


# 9. Generate Predictions for Test Set (Kaggle Submission)


In [ ]:
# Function to generate predictions for test set
def predict_test_set(test_dir, base_models_dict, meta_model, meta_type='lr'):
    """
    Generate predictions for test set using stacking ensemble
    
    Args:
        test_dir: Directory containing test images
        base_models_dict: Dictionary of base models {model_name: model}
        meta_model: Trained meta-learner
        meta_type: Type of meta-learner ('lr', 'xgb', 'nn')
    """
    # Get test image paths
    test_paths = []
    test_ids = []
    
    if os.path.isdir(test_dir):
        # If test_dir is a directory with images
        for img_file in sorted(os.listdir(test_dir)):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                test_paths.append(os.path.join(test_dir, img_file))
                test_ids.append(os.path.splitext(img_file)[0])
    else:
        print(f"Test directory not found: {test_dir}")
        return None, None, None
    
    print(f"Found {len(test_paths)} test images")
    
    # Get predictions from all base models
    test_base_predictions = {}
    for model_name, model in base_models_dict.items():
        print(f"Generating {model_name} predictions...", end=' ')
        test_base_predictions[model_name] = get_predictions_from_model(model, test_paths)
        print("Done")
    
    # Combine predictions
    X_test_stack = combine_predictions(test_base_predictions)
    
    # Get meta-learner predictions
    print(f"Generating final stacking predictions using {meta_type}...", end=' ')
    
    if meta_type == 'lr':
        test_pred_proba = meta_model.predict_proba(X_test_stack)
        test_pred = meta_model.predict(X_test_stack)
    elif meta_type == 'xgb':
        test_pred_proba = meta_model.predict_proba(X_test_stack)
        test_pred = meta_model.predict(X_test_stack)
    elif meta_type == 'nn':
        test_pred_proba = meta_model.predict(X_test_stack, verbose=0)
        test_pred = np.argmax(test_pred_proba, axis=1)
    else:
        raise ValueError(f"Unknown meta_type: {meta_type}")
    
    print("Done")
    
    return test_pred, test_pred_proba, test_ids

# Example usage (uncomment when test directory is available)
# TEST_DIR = '/kaggle/input/car-damage-detection/test'  # Adjust path
# 
# # Load all base models
# base_models_dict = {}
# for model_name in BASE_MODELS:
#     model_path = MODEL_PATHS.get(model_name)
#     base_models_dict[model_name] = load_or_create_model(model_name, num_classes, model_path)
# 
# # Determine meta type
# if best_meta_name == 'Logistic Regression':
#     meta_type = 'lr'
# elif best_meta_name == 'XGBoost':
#     meta_type = 'xgb'
# else:
#     meta_type = 'nn'
# 
# # Generate predictions
# test_predictions, test_probabilities, test_ids = predict_test_set(
#     TEST_DIR, base_models_dict, best_meta_model, meta_type
# )
# 
# # Create submission file
# submission = pd.DataFrame({
#     'id': test_ids,
#     'prediction': [class_names[pred] for pred in test_predictions]
# })
# 
# # Add probability columns if needed
# for i, class_name in enumerate(class_names):
#     submission[f'prob_{class_name}'] = test_probabilities[:, i]
# 
# submission.to_csv('submission.csv', index=False)
# print("Submission file saved as 'submission.csv'")


In [ ]:
# Save meta-learner
import joblib

# Save best meta-learner
if best_meta_name == 'Logistic Regression':
    joblib.dump(best_meta_model, 'meta_learner_lr.pkl')
    print("Saved Logistic Regression meta-learner as 'meta_learner_lr.pkl'")
elif best_meta_name == 'XGBoost':
    joblib.dump(best_meta_model, 'meta_learner_xgb.pkl')
    print("Saved XGBoost meta-learner as 'meta_learner_xgb.pkl'")
elif best_meta_name == 'Neural Network':
    best_meta_model.save('meta_learner_nn.h5')
    print("Saved Neural Network meta-learner as 'meta_learner_nn.h5'")

# Save label encoder
joblib.dump(label_encoder, 'label_encoder.pkl')
print("Saved label encoder as 'label_encoder.pkl'")

# Save predictions for later use (optional)
np.save('train_oof_predictions.npy', train_oof_predictions)
np.save('val_predictions.npy', val_predictions)
print("Saved predictions for later use")


# 11. Advanced: Weighted Average Ensemble (Alternative Approach)


In [ ]:
# Alternative: Weighted average of base model predictions
# This can sometimes work better than stacking

def weighted_average_ensemble(val_predictions_dict, weights=None):
    """
    Create weighted average ensemble
    
    Args:
        val_predictions_dict: Dictionary of predictions from base models
        weights: Optional weights for each model (default: equal weights)
    """
    if weights is None:
        weights = {model: 1.0 / len(BASE_MODELS) for model in BASE_MODELS}
    
    # Initialize with zeros
    ensemble_pred = np.zeros_like(list(val_predictions_dict.values())[0])
    
    # Weighted sum
    for model_name in BASE_MODELS:
        ensemble_pred += weights[model_name] * val_predictions_dict[model_name]
    
    return ensemble_pred

# Try equal weights first
equal_weights = {model: 1.0 / len(BASE_MODELS) for model in BASE_MODELS}
weighted_pred_proba = weighted_average_ensemble(val_predictions, equal_weights)
weighted_pred = np.argmax(weighted_pred_proba, axis=1)
weighted_acc = accuracy_score(y_val_stack, weighted_pred)

print(f"Weighted Average Ensemble (equal weights): {weighted_acc:.4f}")

# Optimize weights using validation performance
from scipy.optimize import minimize

def objective(weights):
    """Objective function to minimize (negative accuracy)"""
    weights_dict = {BASE_MODELS[i]: weights[i] for i in range(len(BASE_MODELS))}
    pred_proba = weighted_average_ensemble(val_predictions, weights_dict)
    pred = np.argmax(pred_proba, axis=1)
    acc = accuracy_score(y_val_stack, pred)
    return -acc  # Minimize negative accuracy

# Constraint: weights sum to 1
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
bounds = [(0, 1) for _ in BASE_MODELS]
initial_weights = [1.0 / len(BASE_MODELS)] * len(BASE_MODELS)

result = minimize(objective, initial_weights, method='SLSQP', 
                  bounds=bounds, constraints=constraints)

optimized_weights = {BASE_MODELS[i]: result.x[i] for i in range(len(BASE_MODELS))}
optimized_pred_proba = weighted_average_ensemble(val_predictions, optimized_weights)
optimized_pred = np.argmax(optimized_pred_proba, axis=1)
optimized_acc = accuracy_score(y_val_stack, optimized_pred)

print(f"\nOptimized Weights:")
for model, weight in optimized_weights.items():
    print(f"  {model}: {weight:.4f}")
print(f"\nWeighted Average Ensemble (optimized): {optimized_acc:.4f}")

# Compare with stacking
print(f"\nStacking ({best_meta_name}): {best_meta_acc:.4f}")
print(f"Weighted Average (optimized): {optimized_acc:.4f}")


# Summary and Notes
# ===========================================
# 
# **Key Points:**
# 1. This notebook implements ensemble stacking with cross-validation to prevent overfitting
# 2. Multiple meta-learners are tested (Logistic Regression, XGBoost, Neural Network)
# 3. The best meta-learner is automatically selected based on validation performance
# 4. An alternative weighted average approach is also provided
# 
# **For Kaggle Submission:**
# - Adjust paths in Section 2 to match your Kaggle dataset structure
# - Uncomment and modify Section 9 to generate test predictions
# - Ensure all base model weights are available or models will be trained from scratch
# 
# **Tips:**
# - Use pre-trained base models for faster execution
# - Adjust N_SPLITS based on dataset size (5-10 folds recommended)
# - Experiment with different meta-learners and their hyperparameters
# - Consider blending stacking and weighted average predictions
